# Module 4: Frozen RoBERTa Embedding Baseline

**Question:** Can a fully frozen RoBERTa-base encoder plus a linear classifier outperform
the leakage-safe TF-IDF baseline on 77 banking intents?

**Evidence requirements**

- pin and hash the encoder snapshot;
- prove that all 124,055,040 encoder parameters remain frozen;
- extract train and validation embeddings before any test embedding;
- record validation-driven search amendments before test access;
- compare models on identical test source indices without persisting messages.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

current = Path.cwd().resolve()
PROJECT_ROOT = next(
    path
    for path in [current, *current.parents]
    if (path / 'pyproject.toml').exists()
)
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
PROJECT_ROOT


## Validate the complete evidence chain

These checks bind the reports to the dataset manifest, configuration, implementation, pinned
model files, selection lock and index-aligned predictions. They do not load message text.


In [ ]:
from governed_banking.data import sha256_file, stable_json_sha256, validate_manifest
from governed_banking.frozen_baseline import (
    FrozenBaselineConfig,
    validate_frozen_evaluation_artifact,
    validate_frozen_selection_artifact,
)

CONFIG_PATH = PROJECT_ROOT / 'configs' / 'frozen_roberta.yaml'
MANIFEST_PATH = PROJECT_ROOT / 'data/manifests/banking77-seed-42.json'
REPORT_DIR = PROJECT_ROOT / 'reports/frozen-roberta'
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
selection = json.loads((REPORT_DIR / 'selection.json').read_text(encoding='utf-8'))
evaluation = json.loads((REPORT_DIR / 'test.json').read_text(encoding='utf-8'))
extraction = json.loads(
    (REPORT_DIR / 'embedding-extraction.json').read_text(encoding='utf-8')
)
comparison = json.loads(
    (REPORT_DIR / 'paired-vs-tfidf.json').read_text(encoding='utf-8')
)
tfidf = json.loads(
    (PROJECT_ROOT / 'reports/baseline/tfidf-logreg-test.json').read_text(
        encoding='utf-8'
    )
)
validate_manifest(manifest)
config = FrozenBaselineConfig.from_yaml(CONFIG_PATH)


In [ ]:
implementation_sha256 = {
    'baseline.py': sha256_file(PROJECT_ROOT / 'src/governed_banking/baseline.py'),
    'data.py': sha256_file(PROJECT_ROOT / 'src/governed_banking/data.py'),
    'device.py': sha256_file(PROJECT_ROOT / 'src/governed_banking/device.py'),
    'frozen_baseline.py': sha256_file(
        PROJECT_ROOT / 'src/governed_banking/frozen_baseline.py'
    ),
    'run_frozen_roberta_baseline.py': sha256_file(
        PROJECT_ROOT / 'scripts/run_frozen_roberta_baseline.py'
    ),
}
model_hashes = selection['encoder']['files_sha256']
validate_frozen_selection_artifact(
    selection,
    dataset_manifest_sha256=manifest['manifest_sha256'],
    config_sha256=sha256_file(CONFIG_PATH),
    implementation_sha256=implementation_sha256,
    model_files_sha256=model_hashes,
)
validate_frozen_evaluation_artifact(
    evaluation,
    selection_sha256=selection['selection_sha256'],
    dataset_manifest_sha256=manifest['manifest_sha256'],
    config_sha256=sha256_file(CONFIG_PATH),
    implementation_sha256=implementation_sha256,
    model_files_sha256=model_hashes,
)
extraction_body = dict(extraction)
assert stable_json_sha256(
    {key: value for key, value in extraction_body.items() if key != 'evidence_sha256'}
) == extraction['evidence_sha256']
comparison_body = dict(comparison)
assert stable_json_sha256(
    {key: value for key, value in comparison_body.items() if key != 'comparison_sha256'}
) == comparison['comparison_sha256']
print('Module 4 evidence chain validated.')


## Registered search and amendments

The initial validation sweep ended at its regularisation boundary. The search was extended
logarithmically while the test split and test embeddings remained inaccessible. Round 4 was
declared final; its interior winner became the lock.


In [ ]:
candidate_results = pd.DataFrame(
    [
        {
            'rank': result['validation_rank'],
            'candidate': result['candidate_name'],
            'pooling': result['candidate']['pooling'],
            'C': result['candidate']['c_value'],
            'macro_f1': result['metrics']['macro_f1'],
            'accuracy': result['metrics']['accuracy'],
            'log_loss': result['metrics']['log_loss'],
            'converged': result['converged'],
        }
        for result in selection['candidate_results']
    ]
).sort_values('rank')
candidate_results


In [ ]:
mean_results = candidate_results[candidate_results['pooling'] == 'mean'].sort_values('C')
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(
    np.log10(mean_results['C']),
    mean_results['macro_f1'],
    marker='o',
    color='#5b8c3a',
)
ax.axvline(np.log10(1024), color='#d0643b', linestyle='--', label='locked winner')
ax.set_title('Validation macro-F1 for frozen mean embeddings')
ax.set_xlabel('log10(C)')
ax.set_ylabel('Macro-F1')
ax.legend()
plt.tight_layout()


## MPS extraction evidence

RoBERTa runs on the Mac GPU in inference mode. Logistic regression runs on one CPU thread.
No BANKING77 row exceeded the registered 96-token limit.


In [ ]:
pd.DataFrame(
    [
        {
            'split': split_name,
            'rows': details['rows'],
            'device': details['device']['selected'],
            'seconds': details['extraction_seconds'],
            'rows_per_second': details['rows_per_second'],
            'truncated_rows': details['token_length']['truncated_rows'],
            'trainable_encoder_parameters': details['trainable_encoder_parameters'],
        }
        for split_name, details in extraction['splits'].items()
    ]
)


## Locked test comparison

The frozen model is evaluated on the same 3,080 source indices as TF-IDF. Higher log-loss
or confidence quality is not interpreted as calibration evidence in this module.


In [ ]:
frozen_metrics = evaluation['test_result']['metrics']
tfidf_metrics = tfidf['test_result']['metrics']
pd.DataFrame(
    {
        'TF-IDF': {name: tfidf_metrics[name] for name in (
            'accuracy', 'macro_f1', 'weighted_f1', 'log_loss', 'top_3_accuracy'
        )},
        'Frozen RoBERTa': {name: frozen_metrics[name] for name in (
            'accuracy', 'macro_f1', 'weighted_f1', 'log_loss', 'top_3_accuracy'
        )},
    }
)


In [ ]:
paired_summary = {**comparison['correctness'], **comparison['metrics']}
pd.Series(paired_summary, name='paired_test_comparison')


## Failure analysis


In [ ]:
worst_intents = (
    pd.DataFrame.from_dict(frozen_metrics['per_intent'], orient='index')
    .rename_axis('intent')
    .sort_values(['f1', 'recall'])
    .head(10)
)
worst_intents


In [ ]:
pd.DataFrame(comparison['candidate_advantage_intents']).head(10)


## Interpretation

- Frozen RoBERTa reaches **0.8964 macro-F1**, below TF-IDF at **0.9053**.
- The paired exact McNemar p-value is 0.1177; this does not establish equivalence.
- RoBERTa fixes 135 rows that TF-IDF misses, while TF-IDF alone fixes 163 rows.
- The models disagree on 368 rows, indicating useful complementarity for later analysis.
- CLS pooling is unsuitable here; final-layer content-token mean pooling is far stronger.
- Generic frozen representations are not a substitute for domain adaptation.
- Module 5 must evaluate whether LoRA improves semantic separation under the same contract.

To reproduce locally, run python scripts/run_frozen_roberta_baseline.py run --offline.
Use --force-embeddings only when intentionally replacing verified local caches.
